# 04 — Stage 1 with Class Weighting

Sensitivity analysis for the 1:1:2 class ratio of stage 1. Only the stage-1 models are retrained, with class-weighted cross-entropy; every other setting, including the seeds, is identical to notebook 01. The flat and stage-2 models from notebooks 01–02 are reused. All settings below were fixed before training.

## 1. Configuration

In [1]:
import torch
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# Paths (adjust to your Google Drive)
DATA_DIR  = Path('/content/drive/MyDrive/USK-Coffee')
LOCAL_DIR = Path('/content/USK-Coffee')
OUT_ROOT  = Path('/content/drive/MyDrive/coffee-two-stage/models')                 # models from notebook 01
W_ROOT    = Path('/content/drive/MyDrive/coffee-two-stage/models_class_weighted')  # new stage-1 models
TEST_OUT  = Path('/content/drive/MyDrive/coffee-two-stage/test_results_class_weighted')

# Data
CLASSES = ['defect', 'longberry', 'peaberry', 'premium']
EXPECTED_COUNT = {'train': 1200, 'val': 400, 'test': 400}
IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Training (same as notebook 01)
BATCH_SIZE = 32
LR = 1e-4
STEP_SIZE, GAMMA = 5, 0.5
MAX_EPOCHS = 30
PATIENCE = 5
NUM_WORKERS = 2

ARCHS = ['resnet50', 'efficientnet_b0']
SEEDS = [0, 1, 2]
ALPHA = 0.05

# Stage-1 training images: defect 1200, longberry 1200, merged 2400
# weight = n_total / (n_classes * n_class), so each class contributes equally to the loss
STAGE1_COUNTS = [1200, 1200, 2400]
CLASS_WEIGHTS = torch.tensor([sum(STAGE1_COUNTS) / (3 * n) for n in STAGE1_COUNTS])
print('class weights:', CLASS_WEIGHTS.tolist())

def run_dir(arch, scheme, seed):
    return OUT_ROOT / arch / scheme / f'seed{seed}'

def w_dir(arch, seed):
    return W_ROOT / arch / '3class_weighted' / f'seed{seed}'

TEST_OUT.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive
class weights: [1.3333333730697632, 1.3333333730697632, 0.6666666865348816]


## 2. Copy data to the Colab disk

In [2]:
import shutil, time

EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

def list_images(folder):
    return sorted(f for f in folder.iterdir() if f.suffix.lower() in EXTS)

t0 = time.time()
for split in ['train', 'val', 'test']:
    dst = LOCAL_DIR / split
    ok = dst.exists() and all(
        (dst / c).exists() and len(list_images(dst / c)) == EXPECTED_COUNT[split]
        for c in CLASSES)
    if not ok:
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(DATA_DIR / split, dst)
    for c in CLASSES:
        assert len(list_images(dst / c)) == EXPECTED_COUNT[split], f'{split}/{c} incomplete'
print(f'done in {time.time() - t0:.0f}s')

done in 128s


## 3. Dataset and transforms

In [3]:
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SCHEMES_MAP = {
    '4class': {'defect': 0, 'longberry': 1, 'peaberry': 2, 'premium': 3},
    '3class': {'defect': 0, 'longberry': 1, 'peaberry': 2, 'premium': 2},
}

# Same augmentation as notebook 01
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class CoffeeDataset(Dataset):
    def __init__(self, split_dir, scheme, transform):
        self.samples = [(f, label) for c, label in SCHEMES_MAP[scheme].items()
                        for f in list_images(split_dir / c)]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        with Image.open(path) as img:
            return self.transform(img.convert('RGB')), label

## 4. Training function

In [4]:
import pandas as pd
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(_):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

def build_model(arch, num_classes, pretrained=True):
    if arch == 'resnet50':
        m = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif arch == 'efficientnet_b0':
        m = models.efficientnet_b0(weights='IMAGENET1K_V1' if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    else:
        raise ValueError(arch)
    return m

@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        probs.append(torch.softmax(model(x.to(device)), 1).cpu())
        labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()

def epochs_since_improvement(history):
    losses = [h['val_loss'] for h in history]
    return len(losses) - 1 - int(np.argmin(losses))

def should_stop(history):
    return len(history) >= MAX_EPOCHS or (len(history) > 0 and epochs_since_improvement(history) >= PATIENCE)

def train_stage1_weighted(arch, seed, out_dir, pretrained=True):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path  = out_dir / 'last_checkpoint.pt'
    best_path  = out_dir / 'best_model.pt'
    final_path = out_dir / 'final_model.pt'
    if final_path.exists():
        print(f'skip (done): {out_dir}')
        return

    set_seed(seed)
    train_ds = CoffeeDataset(LOCAL_DIR / 'train', '3class', train_tf)
    val_ds   = CoffeeDataset(LOCAL_DIR / 'val', '3class', eval_tf)
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                              worker_init_fn=seed_worker, generator=g, pin_memory=True)
    val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = build_model(arch, 3, pretrained).to(device)
    # The only change from notebook 01: class-weighted loss
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)

    history = []
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler']); history = ck['history']
        g.set_state(ck['rng_gen']); torch.set_rng_state(ck['rng_torch'])
        np.random.set_state(ck['rng_np']); random.setstate(ck['rng_py'])
        if ck['rng_cuda'] is not None: torch.cuda.set_rng_state_all(ck['rng_cuda'])
        print(f'resuming from epoch {len(history) + 1}')

    while not should_stop(history):
        epoch = len(history) + 1
        t0 = time.time()
        lr = optimizer.param_groups[0]['lr']
        model.train()
        loss_sum, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item(); total += len(y)
        scheduler.step()

        prob, y_val = predict(model, val_loader)
        # Validation loss uses the same class weights (validation is also 1:1:2)
        val_loss = nn.functional.nll_loss(torch.log(torch.tensor(prob) + 1e-12), torch.tensor(y_val),
                                          weight=CLASS_WEIGHTS).item()
        row = {'epoch': epoch, 'lr': lr, 'train_loss': loss_sum / total, 'train_acc': correct / total,
               'val_loss': val_loss, 'val_acc': float((prob.argmax(1) == y_val).mean()), 'sec': time.time() - t0}

        if row['val_loss'] < min((h['val_loss'] for h in history), default=float('inf')):
            torch.save(model.state_dict(), best_path)
        history.append(row)

        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'history': history,
                    'rng_gen': g.get_state(), 'rng_torch': torch.get_rng_state(),
                    'rng_np': np.random.get_state(), 'rng_py': random.getstate(),
                    'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
                   ckpt_path)
        pd.DataFrame(history).to_csv(out_dir / 'history.csv', index=False)
        print(f"  epoch {epoch:>2}/{MAX_EPOCHS} | train {row['train_loss']:.4f}/{row['train_acc']:.4f} "
              f"| val {val_loss:.4f}/{row['val_acc']:.4f} | {row['sec']:.0f}s")

    hist = pd.DataFrame(history)
    best = hist.loc[hist['val_loss'].idxmin()]
    print(f"  stopped after epoch {len(hist)} | best val_loss {best['val_loss']:.4f} "
          f"(val_acc {best['val_acc']:.4f}) at epoch {int(best['epoch'])}")

    model.load_state_dict(torch.load(best_path, map_location=device))
    full_val = CoffeeDataset(LOCAL_DIR / 'val', '4class', eval_tf)
    prob, y_true = predict(model, DataLoader(full_val, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS))
    np.savez(out_dir / 'val_predictions.npz', prob=prob, y_true=y_true,
             paths=np.array([str(p.relative_to(LOCAL_DIR)) for p, _ in full_val.samples]))

    best_path.replace(final_path)
    ckpt_path.unlink(missing_ok=True)

## 5. Train the 6 stage-1 models

In [5]:
# Completed runs are skipped, interrupted runs resume
runs = [(seed, arch) for seed in SEEDS for arch in ARCHS]

for i, (seed, arch) in enumerate(runs, 1):
    print(f'[{i}/{len(runs)}] {arch} | 3class_weighted | seed {seed}')
    train_stage1_weighted(arch, seed, w_dir(arch, seed))
print('All runs finished')

## 6. Training status

In [6]:
status = []
for seed, arch in runs:
    d = w_dir(arch, seed)
    row = {'arch': arch, 'seed': seed}
    if (d / 'history.csv').exists():
        h = pd.read_csv(d / 'history.csv')
        b = h.loc[h['val_loss'].idxmin()]
        row.update({'epochs_run': len(h), 'best_epoch': int(b['epoch']),
                    'best_val_loss': round(b['val_loss'], 4), 'val_acc_at_best': round(b['val_acc'], 4),
                    'train_minutes': round(h['sec'].sum() / 60, 1)})
    row['status'] = ('done' if (d / 'final_model.pt').exists()
                     else 'running/interrupted' if (d / 'history.csv').exists() else 'not started')
    status.append(row)
status = pd.DataFrame(status)
status.to_csv(TEST_OUT / 'training_status_weighted.csv', index=False)
print(status.to_string(index=False))

           arch  seed  epochs_run  best_epoch  best_val_loss  val_acc_at_best  train_minutes status
       resnet50     0          11           6         0.1426           0.9550           12.4   done
efficientnet_b0     0          11           6         0.1625           0.9506            9.9   done
       resnet50     1          17          12         0.1639           0.9594           19.4   done
efficientnet_b0     1          18          13         0.1889           0.9419           16.7   done
       resnet50     2          13           8         0.1017           0.9619           15.1   done
efficientnet_b0     2          13           8         0.1737           0.9375           11.7   done


## 7. Test predictions

In [7]:
test_ds = CoffeeDataset(LOCAL_DIR / 'test', '4class', eval_tf)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_paths = np.array([str(p.relative_to(LOCAL_DIR)) for p, _ in test_ds.samples])

for seed, arch in runs:
    d = w_dir(arch, seed)
    f = d / 'test_predictions.npz'
    if f.exists():
        print(f'skip (already predicted): {d}')
        continue
    model = build_model(arch, 3, pretrained=False)
    model.load_state_dict(torch.load(d / 'final_model.pt', map_location='cpu'))
    prob, y_true = predict(model.to(device), test_loader)
    np.savez(f, prob=prob, y_true=y_true, paths=test_paths)
    print(f'predicted: {d}')

## 8. Metrics per seed

In [8]:
from sklearn.metrics import precision_recall_fscore_support

def load_npz(path):
    z = np.load(path)
    assert (z['paths'] == test_paths).all(), f'image order mismatch: {path}'
    return z

# Merged-class images go to stage 2; others keep the stage-1 label
def hierarchical_predict(s1_prob, s2_prob):
    s1 = s1_prob.argmax(1)
    s2 = s2_prob.argmax(1)
    return np.where(s1 == 2, 2 + s2, s1)

def metrics_row(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=range(4), zero_division=0)
    errors = (y_true != y_pred).sum()
    pb_to_pr = ((y_true == 2) & (y_pred == 3)).sum()
    pr_to_pb = ((y_true == 3) & (y_pred == 2)).sum()
    row = {'accuracy': (y_true == y_pred).mean(), 'macro_f1': f1.mean(), 'errors': errors,
           'pair_errors': pb_to_pr + pr_to_pb, 'other_errors': errors - pb_to_pr - pr_to_pb}
    for k, c in enumerate(CLASSES):
        row.update({f'P_{c}': p[k], f'R_{c}': r[k], f'F1_{c}': f1[k]})
    return row

def stage_details(y, s1_prob, s2_prob):
    s1_pred = s1_prob.argmax(1)
    pair = np.isin(y, [2, 3])
    return {'stage1_acc': (np.where(y >= 2, 2, y) == s1_pred).mean(),
            'stage2_acc_on_pair': (s2_prob.argmax(1)[pair] == y[pair] - 2).mean(),
            'pair_lost_at_stage1': (pair & (s1_pred != 2)).sum(),
            'other_routed_to_stage2': (~pair & (s1_pred == 2)).sum(),
            'routed_to_stage2_share': (s1_pred == 2).mean()}

rows, preds = [], {}
for arch in ARCHS:
    for seed in SEEDS:
        flat = load_npz(run_dir(arch, '4class', seed) / 'test_predictions.npz')
        s1   = load_npz(run_dir(arch, '3class', seed) / 'test_predictions.npz')
        s2   = load_npz(run_dir(arch, '2class', seed) / 'test_predictions.npz')
        s1w  = load_npz(w_dir(arch, seed) / 'test_predictions.npz')
        y = flat['y_true']
        pred_flat = flat['prob'].argmax(1)
        pred_hier = hierarchical_predict(s1['prob'], s2['prob'])
        # Weighted stage 1 combined with the original stage 2
        pred_hier_w = hierarchical_predict(s1w['prob'], s2['prob'])
        preds[(arch, seed)] = (y, pred_flat, pred_hier, pred_hier_w)

        rows.append({'arch': arch, 'seed': seed, 'model': 'flat', **metrics_row(y, pred_flat)})
        rows.append({'arch': arch, 'seed': seed, 'model': 'hierarchical',
                     **metrics_row(y, pred_hier), **stage_details(y, s1['prob'], s2['prob'])})
        rows.append({'arch': arch, 'seed': seed, 'model': 'hierarchical_weighted',
                     **metrics_row(y, pred_hier_w), **stage_details(y, s1w['prob'], s2['prob'])})

results = pd.DataFrame(rows)
results.to_csv(TEST_OUT / 'test_results_per_seed_weighted.csv', index=False)

main_cols = ['accuracy', 'macro_f1', 'pair_errors', 'other_errors']
stage_cols = ['stage1_acc', 'stage2_acc_on_pair', 'pair_lost_at_stage1',
              'other_routed_to_stage2', 'routed_to_stage2_share']
pd.set_option('display.width', 220)
print(results[['arch', 'model', 'seed'] + main_cols].round(4).to_string(index=False))
print('\nStage details:')
print(results[results['model'] != 'flat'][['arch', 'model', 'seed'] + stage_cols].round(4).to_string(index=False))

           arch                 model  seed  accuracy  macro_f1  pair_errors  other_errors
       resnet50                  flat     0    0.8019    0.8002          112           205
       resnet50          hierarchical     0    0.8162    0.8201           83           211
       resnet50 hierarchical_weighted     0    0.8488    0.8508           82           160
       resnet50                  flat     1    0.7950    0.8002           93           235
       resnet50          hierarchical     1    0.8662    0.8665           41           173
       resnet50 hierarchical_weighted     1    0.8450    0.8417           42           206
       resnet50                  flat     2    0.8725    0.8735           64           140
       resnet50          hierarchical     2    0.8281    0.8306           62           213
       resnet50 hierarchical_weighted     2    0.8638    0.8647           61           157
efficientnet_b0                  flat     0    0.8250    0.8278          111           169

## 9. Mean and standard deviation

In [9]:
summary = results.groupby(['arch', 'model'], sort=False)[main_cols + stage_cols].agg(['mean', 'std'])
summary.to_csv(TEST_OUT / 'test_results_summary_weighted.csv')
print(summary.round(4).to_string())

                                      accuracy         macro_f1         pair_errors          other_errors          stage1_acc         stage2_acc_on_pair         pair_lost_at_stage1          other_routed_to_stage2          routed_to_stage2_share        
                                          mean     std     mean     std        mean      std         mean      std       mean     std               mean     std                mean      std                   mean      std                   mean     std
arch            model                                                                                                                                                                                                                                       
resnet50        flat                    0.8231  0.0429   0.8246  0.0423     89.6667  24.1730     193.3333  48.5627        NaN     NaN                NaN     NaN                 NaN      NaN                    NaN      NaN                    

## 10. McNemar test (weighted two-stage vs flat)

In [10]:
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

# Same procedure as notebook 02
tests = []
for arch in ARCHS:
    for seed in SEEDS:
        y, pf, _, phw = preds[(arch, seed)]
        for subset, mask in [('all_1600', np.ones_like(y, bool)), ('pair_800', np.isin(y, [2, 3]))]:
            cf, ch = (pf == y)[mask], (phw == y)[mask]
            b = int((cf & ~ch).sum())   # correct by flat only
            c = int((~cf & ch).sum())   # correct by weighted two-stage only
            table = [[int((cf & ch).sum()), b], [c, int((~cf & ~ch).sum())]]
            direction = 'hier_better' if c > b else ('flat_better' if b > c else 'equal')
            tests.append({'arch': arch, 'seed': seed, 'subset': subset,
                          'acc_flat': cf.mean(), 'acc_hier_w': ch.mean(),
                          'b_flat_only': b, 'c_hier_w_only': c, 'direction': direction,
                          'p_exact': mcnemar(table, exact=True).pvalue})

mc = pd.DataFrame(tests)
mc['p_holm'] = np.nan
for arch in ARCHS:
    idx = mc['arch'] == arch
    mc.loc[idx, 'p_holm'] = multipletests(mc.loc[idx, 'p_exact'], alpha=ALPHA, method='holm')[1]
mc['significant'] = mc['p_holm'] < ALPHA
mc.to_csv(TEST_OUT / 'mcnemar_results_weighted.csv', index=False)
print(mc.round(4).to_string(index=False))

rows = []
for (arch, subset), g in mc.groupby(['arch', 'subset'], sort=False):
    n_hier = int((g['significant'] & (g['direction'] == 'hier_better')).sum())
    n_flat = int((g['significant'] & (g['direction'] == 'flat_better')).sum())
    if n_hier == len(SEEDS):
        verdict = 'different: hierarchical better (all seeds)'
    elif n_flat == len(SEEDS):
        verdict = 'different: flat better (all seeds)'
    elif n_hier + n_flat > 0:
        verdict = 'inconsistent across seeds'
    else:
        verdict = 'no significant difference'
    rows.append({'arch': arch, 'subset': subset, 'sig_hier_w_better': f'{n_hier}/{len(SEEDS)}',
                 'sig_flat_better': f'{n_flat}/{len(SEEDS)}', 'verdict': verdict})
conclusion = pd.DataFrame(rows)
conclusion.to_csv(TEST_OUT / 'mcnemar_conclusion_weighted.csv', index=False)
print(conclusion.to_string(index=False))

           arch  seed   subset  acc_flat  acc_hier_w  b_flat_only  c_hier_w_only   direction  p_exact  p_holm  significant
       resnet50     0 all_1600    0.8019      0.8488           82            157 hier_better   0.0000  0.0000         True
       resnet50     0 pair_800    0.8212      0.8462           44             64 hier_better   0.0670  0.1340        False
       resnet50     1 all_1600    0.7950      0.8450           78            158 hier_better   0.0000  0.0000         True
       resnet50     1 pair_800    0.8700      0.9212           24             65 hier_better   0.0000  0.0001         True
       resnet50     2 all_1600    0.8725      0.8638           96             82 flat_better   0.3299  0.3299        False
       resnet50     2 pair_800    0.8925      0.8675           51             31 flat_better   0.0352  0.1057        False
efficientnet_b0     0 all_1600    0.8250      0.8638           48            110 hier_better   0.0000  0.0000         True
efficientnet_b0 